In [ ]:
# ============================================================
# ASSIGNMENT 2 - Q5: PER-RERANKER x PER-SLICE COMPARISON (MIND, dev)
# Team: Shivam Patel + Suyash Pande | CS4.406 IRE
#
# Compares ALL rerankers across the SAME slices, on the SAME dev impressions:
#   rerankers: LightGBM, MLP, NRMS(title-only baseline), NRMS(title+abstract)
#   slices:    all / cold vs warm (user history length, p20/p80) / head vs tail (clicked-article popularity)
# Metric: per-impression AUC (mean per slice) + bootstrap 95% CI.
#
# Answers: does any model handle COLD users better than feature-GBDT?
#   (content-based neural models sometimes help cold-start where behavioural features are sparse)
#
# All from cached scores on HF (no training):
#   mind_nrms_devscores.pkl, mind_nrms_ta_devscores.pkl, mind_large_features.npz (+iids_dv)
# NEEDS: Kaggle MIND-large mounted (for labels + history lengths + article popularity).
# ============================================================
!pip install lightgbm huggingface_hub -q
import glob, numpy as np, pandas as pd, pickle
from bisect import bisect_left
from collections import defaultdict
import lightgbm as lgb, torch, torch.nn as nn
from huggingface_hub import hf_hub_download
HF_REPO="donbosoc/mind-artifacts"
def dl(name): return hf_hub_download(HF_REPO, name, repo_type="dataset")
LTR=glob.glob("/kaggle/input/**/MINDlarge_train",recursive=True)[0]
LDV=glob.glob("/kaggle/input/**/MINDlarge_dev",recursive=True)[0]
BEH=["impression_id","user_id","time","history","impressions"]
def pfx(x): return f"mind:{x}"

def _auc(s,lb):
    p=lb==1;n=lb==0;np_,nn=p.sum(),n.sum()
    if np_==0 or nn==0: return None
    o=np.argsort(s);r=np.empty_like(o,float);r[o]=np.arange(1,len(s)+1)
    return float((r[p].sum()-np_*(np_+1)/2)/(np_*nn))


In [ ]:
# ---- dev labels/slate + per-user history length + article popularity (for slicing) ----
def load_beh(p):
    return pd.read_csv(f"{p}/behaviors.tsv",sep="\t",header=None,names=BEH,quoting=3)
b_tr=load_beh(LTR); b_dv=load_beh(LDV)
# history length per user
hist_len={}
for b in (b_tr,b_dv):
    for u,h in zip(b["user_id"],b["history"]):
        if isinstance(h,str): hist_len[pfx(u)]=len(h.split()) if h else 0
# article click popularity (train+dev)
pop=defaultdict(int)
for b in (b_tr,b_dv):
    for imps in b["impressions"]:
        if isinstance(imps,str):
            for tk in imps.split():
                if tk.endswith("-1"): pop[pfx(tk.split("-")[0])]+=1
# dev slate: impression_id -> (user_id, [(article,label)])
imp_slate={}; imp_user={}
for iid,u,imps in zip(b_dv["impression_id"],b_dv["user_id"],b_dv["impressions"]):
    if not isinstance(imps,str): continue
    slate=[]
    for tk in imps.split():
        pp=tk.split("-")
        if len(pp)==2: slate.append((pfx(pp[0]),int(pp[1])))
    if slate: imp_slate[iid]=slate; imp_user[iid]=pfx(u)
print("dev impressions with slate:",len(imp_slate))


In [ ]:
# ---- gather per-impression AUC for each reranker, keyed by impression_id ----
# NRMS-family from pkl caches; GBDT/MLP from npz (keyed by iids_dv)
def per_imp_auc_from_cache(cache):
    by_imp={}
    for (iid,aid),sc in cache.items(): by_imp.setdefault(iid,{})[aid]=sc
    out={}
    for iid,scores in by_imp.items():
        slate=imp_slate.get(iid)
        if not slate: continue
        s=[];lb=[]
        for aid,lab in slate:
            if aid in scores: s.append(scores[aid]); lb.append(lab)
        if len(s)>=2 and sum(lb)>0:
            a=_auc(np.array(s),np.array(lb))
            if a is not None: out[iid]=a
    return out

nrms_base = per_imp_auc_from_cache(pickle.load(open(dl("mind_nrms_devscores.pkl"),"rb")))
nrms_ta   = per_imp_auc_from_cache(pickle.load(open(dl("mind_nrms_ta_devscores.pkl"),"rb")))
print("NRMS base",len(nrms_base),"| NRMS ta",len(nrms_ta))

# GBDT + MLP from npz (need iids_dv)
d=np.load(dl("mind_large_features.npz"),allow_pickle=True)
Xdv,ydv,gdv=d["Xdv"].astype(np.float32),d["ydv"],[int(g) for g in d["gdv"]]
iids_dv=list(d["iids_dv"]) if "iids_dv" in d.files else None
FEAT=[str(f) for f in d["feat"]]
lgbm=lgb.Booster(model_file=dl("mind_lgbm_reranker.txt"))
lgb_sc=lgbm.predict(Xdv)

class MLP(nn.Module):
    def __init__(s,din):
        super().__init__(); s.net=nn.Sequential(nn.Linear(din,128),nn.ReLU(),nn.Dropout(0.2),
                                                 nn.Linear(128,64),nn.ReLU(),nn.Linear(64,1))
    def forward(s,x): return s.net(x).squeeze(-1)
mlp=MLP(len(FEAT)); mlp.load_state_dict(torch.load(dl("mind_mlp.pt"),map_location="cpu")); mlp.eval()
with torch.no_grad(): mlp_sc=mlp(torch.tensor(Xdv)).numpy()

def per_imp_auc_from_npz(scores):
    out={}; pos=0
    for gi,iid in zip(gdv, (iids_dv if iids_dv else range(len(gdv)))):
        a=_auc(scores[pos:pos+gi], ydv[pos:pos+gi]); pos+=gi
        if a is not None and iids_dv is not None: out[iid]=a
    return out
gbdt=per_imp_auc_from_npz(lgb_sc); mlp_a=per_imp_auc_from_npz(mlp_sc)
print("GBDT",len(gbdt),"| MLP",len(mlp_a))
if iids_dv is None:
    print("WARNING: npz lacks iids_dv -> GBDT/MLP cannot be sliced by impression. Re-run updated A2_01.")


In [ ]:
# ---- slice each model's per-impression AUC by cold/warm (history) and head/tail (popularity) ----
models={"NRMS-base":nrms_base,"NRMS-ta":nrms_ta,"MLP":mlp_a,"LightGBM":gbdt}

# slice boundaries from history-length distribution over all dev impressions we have labels for
all_iids=set(imp_slate)
hls=np.array([hist_len.get(imp_user[i],0) for i in all_iids])
P20,P80=np.percentile(hls,20),np.percentile(hls,80)
# head/tail by clicked-article popularity (median over dev)
def clicked_pop(iid):
    for aid,lab in imp_slate[iid]:
        if lab==1: return pop.get(aid,0)
    return 0
pops=np.array([clicked_pop(i) for i in all_iids]); POPMED=np.median(pops)
print(f"history p20={P20:.0f} p80={P80:.0f} | clicked-pop median={POPMED:.0f}")

def slice_of_user(iid):
    hl=hist_len.get(imp_user[iid],0)
    return "cold" if hl<=P20 else ("warm" if hl>=P80 else "mid")
def slice_of_art(iid):
    return "head" if clicked_pop(iid)>POPMED else "tail"

def bci(vals,nb=500):
    v=np.array(vals)
    if len(v)==0: return (float('nan'),)*3
    rng=np.random.default_rng(0)
    m=[rng.choice(v,len(v),replace=True).mean() for _ in range(nb)]
    return v.mean(),np.percentile(m,2.5),np.percentile(m,97.5)

def collect(model_auc, slice_fn, slice_name):
    buckets=defaultdict(list)
    for iid,a in model_auc.items():
        if iid in imp_slate:
            buckets[slice_fn(iid)].append(a)
            buckets["all"].append(a)
    return buckets

print("\n=== PER-RERANKER x USER-SLICE (cold/warm) AUC ===")
print(f"{'model':10s} {'all':>18} {'cold':>18} {'warm':>18}")
for name,auc in models.items():
    b=collect(auc, slice_of_user, "user")
    cells=[]
    for sl in ("all","cold","warm"):
        m,lo,hi=bci(b[sl]); cells.append(f"{m:.4f}[{lo:.3f},{hi:.3f}]")
    print(f"{name:10s} {cells[0]:>18} {cells[1]:>18} {cells[2]:>18}")

print("\n=== PER-RERANKER x ARTICLE-SLICE (head/tail) AUC ===")
print(f"{'model':10s} {'all':>18} {'head':>18} {'tail':>18}")
for name,auc in models.items():
    b=collect(auc, slice_of_art, "art")
    cells=[]
    for sl in ("all","head","tail"):
        m,lo,hi=bci(b[sl]); cells.append(f"{m:.4f}[{lo:.3f},{hi:.3f}]")
    print(f"{name:10s} {cells[0]:>18} {cells[1]:>18} {cells[2]:>18}")


In [ ]:
# ---- cold-start focus: which model wins on cold users? ----
print("="*70); print("COLD-START ANALYSIS: does any model beat LightGBM on cold users?"); print("="*70)
cold_scores={}
for name,auc in models.items():
    vals=[a for iid,a in auc.items() if iid in imp_slate and slice_of_user(iid)=="cold"]
    if vals: cold_scores[name]=np.mean(vals)
best=max(cold_scores,key=cold_scores.get)
for name in ("NRMS-base","NRMS-ta","MLP","LightGBM"):
    if name in cold_scores:
        mark=" <- best on cold" if name==best else ""
        print(f"  {name:10s} cold AUC {cold_scores[name]:.4f}{mark}")
print(f"""
Interpretation:
 - If LightGBM still wins cold: behavioural features remain strongest even when history is short
   (popularity/CTR of the CANDIDATE don't need a long user history).
 - If a content model (NRMS-ta) wins cold: content helps most when behavioural user-history
   signal is sparse -> a case for a content-based cold-start path.
""")
